# This code contains the results of running the reward model experiment

In [ ]:
import numpy as np

def bootstrap_significance_ci(baseline_scores, gaze_scores, n_boot=1000, ci=90):
    """
    Computes bootstrap mean difference, single-sided p-value, and confidence interval.
    
    Parameters:
        baseline_scores : list or array
            Accuracy/metric per fold for baseline.
        gaze_scores : list or array
            Accuracy/metric per fold for gaze model (same length).
        n_boot : int
            Number of bootstrap samples.
        ci : float
            Confidence level (default 95).

    Returns:
        mean_diff : float
            Mean of bootstrap differences (gaze - baseline)
        p_value : float
            Single-sided p-value for gaze > baseline
        ci_range : tuple
            Two-sided confidence interval (lower, upper)
    """
    baseline_scores = np.array(baseline_scores)
    gaze_scores = np.array(gaze_scores)
    diffs = []
    rng = np.random.default_rng(42)

    for _ in range(n_boot):
        idx = rng.choice(len(baseline_scores), len(baseline_scores), replace=True)
        diff = np.mean(gaze_scores[idx] - baseline_scores[idx])
        diffs.append(diff)

    diffs = np.array(diffs)
    mean_diff = np.mean(diffs)

    p_value = np.sum(diffs <= 0) / n_boot

    lower = np.percentile(diffs, (100 - ci) / 2)
    upper = np.percentile(diffs, 100 - (100 - ci) / 2)
    ci_range = (lower, upper)

    return mean_diff, p_value, ci_range


In [ ]:
# embedding_generation

import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/data1/huggingface"

MODEL_NAME = '/data1/model/llama3.1-8b-base' 
DATA_PATH = "unique_conversations.csv"               
OUT_PATH = "unique_conversations.csv"       
EMB_DIR = "embeddings"                      
DEVICE = 'cuda'

os.makedirs(EMB_DIR, exist_ok=True)


print("Loading model:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()


@torch.no_grad()
def get_embedding(text: str, pooling: str = "mean") -> np.ndarray:

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    outputs = model.model(**inputs, output_hidden_states=True)
    hidden_states = outputs.hidden_states[-1].squeeze(0)

    if pooling == "mean":
        emb = hidden_states.mean(dim=0)
    elif pooling == "cls":
        emb = hidden_states[0]  # BOS token
    else:
        raise ValueError("pooling must be 'mean' or 'cls'")

    return emb.cpu().numpy().astype(np.float32)


def main():
    df = pd.read_csv(DATA_PATH)
    emb_paths = []

    for i, row in df.iterrows():
        text = str(row["text"])
        emb = get_embedding(text, pooling="mean")

        fname = f"{row['sentenceid']}.npy"
        fpath = os.path.join(EMB_DIR, fname)
        np.save(fpath, emb)

        emb_paths.append(fpath)

        if (i + 1) % 20 == 0:
            print(f"Processed {i+1}/{len(df)}")

    df["embedding_path"] = emb_paths
    df.to_csv(OUT_PATH, index=False)
    print("Done. Saved:", OUT_PATH)


if __name__ == "__main__":
    main()


Loading model: /data1/model/llama3.1-8b-base


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.12s/it]


Processed 20/39
✅ Done. Saved: unique_conversations.csv


In [ ]:
# pca_reduction

import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import joblib


DATA_PATH = "metrics_collection.csv" 
EMB_DIR = "embeddings"                    
OUT_DIR = "embeddings_pca128"          
OUT_CSV = "metrics_collection_pca128.csv"  
PCA_MODEL_PATH = "pca_model_128.pkl"     
N_COMPONENTS = 128                       

os.makedirs(OUT_DIR, exist_ok=True)


df = pd.read_csv(DATA_PATH)
all_embs = []
paths = []

print("Loading embeddings...")
for i, row in df.iterrows():
    emb = np.load(row["embedding_path"]) 
    all_embs.append(emb)
    paths.append(row["embedding_path"])
all_embs = np.vstack(all_embs)  
all_embs = (all_embs - all_embs.mean(axis=0)) / (all_embs.std(axis=0) + 1e-8)

print("Final stacked shape:", all_embs.shape)
print("Any NaN in all_embs?", np.any(np.isnan(all_embs)))
print("Any Inf in all_embs?", np.any(np.isinf(all_embs)))

row_nan = np.where(np.isnan(all_embs).any(axis=1))[0]
row_inf = np.where(np.isinf(all_embs).any(axis=1))[0]

print("Rows with NaN:", row_nan)
print("Rows with Inf:", row_inf)
print("Min:", all_embs.min(), "Max:", all_embs.max(), "Mean:", all_embs.mean(), "Std:", all_embs.std())

print(f"Loaded {all_embs.shape[0]} embeddings, dim={all_embs.shape[1]}")


print("Fitting PCA...")
pca = PCA(n_components=N_COMPONENTS, svd_solver="full", random_state=42)

reduced = pca.fit_transform(all_embs) 


joblib.dump(pca, PCA_MODEL_PATH)
print(f"PCA model saved to {PCA_MODEL_PATH}")


new_paths = []
for i, row in df.iterrows():
    fname = os.path.basename(row["embedding_path"]).replace(".npy", f"_pca{N_COMPONENTS}.npy")
    out_path = os.path.join(OUT_DIR, fname)
    np.save(out_path, reduced[i].astype(np.float32))
    new_paths.append(out_path)


df["embedding_path"] = new_paths
df.to_csv(OUT_CSV, index=False)
print(f"Saved reduced embeddings to {OUT_DIR}")
print(f"Updated CSV saved to {OUT_CSV}")


Loading embeddings...
Final stacked shape: (974, 4096)
Any NaN in all_embs? False
Any Inf in all_embs? False
Rows with NaN: []
Rows with Inf: []
Min: -5.2641177 Max: 5.428317 Mean: 1.126384e-09 Std: 0.99999976
Loaded 974 embeddings, dim=4096
Fitting PCA...
✅ PCA model saved to pca_model_128.pkl
✅ Saved reduced embeddings to embeddings_pca128
✅ Updated CSV saved to metrics_collection_pca128.csv


In [ ]:
# set_seed
import random
import numpy as np
import torch

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
from main_ex import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed", "fixation_time", "N100"]
all_dfs = []
repeats = 4
for r in range(repeats):
    seed = 42 + r 
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=False,

    )
 
    res_df["repeat"] = r + 1
    all_dfs.append(res_df)


df = pd.concat(all_dfs, ignore_index=True)
summary = df.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/4 with seed 42 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.551724    0.551724         0     1
1  0.583333    0.583333         0     2
2  0.647059    0.647059         0     3
3  0.592593    0.592593         0     4
4  0.527778    0.527778         0     5
exp_acc       0.580497
strong_acc    0.580497
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 2/4 with seed 43 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.517241    0.517241         0     1
1  0.750000    0.750000         0     2
2  0.558824    0.558824         0     3
3  0.555556    0.555556         0     4
4  0.583333    0.583333         0     5
exp_acc       0.592991
strong_acc    0.592991
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 3/4 with seed 44 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.620690    0.620690         0     1
1  0.79

In [ ]:
from main_ext import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed", "fixation_time", "N100"]
all_dfs = []
repeats = 4
for r in range(repeats):
    seed = 42 + r 
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,

    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)


df_eye = pd.concat(all_dfs, ignore_index=True)
summary = df_eye.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/4 with seed 42 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.517241    0.517241         0     1
1  0.833333    0.833333         0     2
2  0.705882    0.705882         0     3
3  0.629630    0.629630         0     4
4  0.555556    0.555556         0     5
exp_acc       0.648328
strong_acc    0.648328
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 2/4 with seed 43 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.586207    0.586207         0     1
1  0.666667    0.666667         0     2
2  0.764706    0.764706         0     3
3  0.555556    0.555556         0     4
4  0.638889    0.638889         0     5
exp_acc       0.642405
strong_acc    0.642405
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 3/4 with seed 44 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.620690    0.620690         0     1
1  0.70

In [ ]:
from main_ex import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed", "fixation_time"]
all_dfs = []
repeats = 4
for r in range(repeats):
    seed = 42 + r 
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,

    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)


df_n100 = pd.concat(all_dfs, ignore_index=True)
summary = df_n100.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/4 with seed 42 =====



===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.655172    0.655172         0     1
1  0.791667    0.791667         0     2
2  0.705882    0.705882         0     3
3  0.666667    0.666667         0     4
4  0.611111    0.611111         0     5
exp_acc       0.6861
strong_acc    0.6861
weak_acc      0.0000
fold          3.0000
dtype: float64

===== Running repeat 2/4 with seed 43 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.586207    0.586207         0     1
1  0.625000    0.625000         0     2
2  0.647059    0.647059         0     3
3  0.666667    0.666667         0     4
4  0.666667    0.666667         0     5
exp_acc       0.63832
strong_acc    0.63832
weak_acc      0.00000
fold          3.00000
dtype: float64

===== Running repeat 3/4 with seed 44 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.586207    0.586207         0     1
1  0.583333    0.583333         0     2
2  0.647059    0.647059 

In [ ]:
from main_ex import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed",  "N100"]
all_dfs = []
repeats = 4
for r in range(repeats):
    seed = 42 + r 
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,

    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)

df_ft = pd.concat(all_dfs, ignore_index=True)
summary = df_ft.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/4 with seed 42 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.655172    0.655172         0     1
1  0.791667    0.791667         0     2
2  0.705882    0.705882         0     3
3  0.666667    0.666667         0     4
4  0.583333    0.583333         0     5
exp_acc       0.680544
strong_acc    0.680544
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 2/4 with seed 43 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.620690    0.620690         0     1
1  0.625000    0.625000         0     2
2  0.676471    0.676471         0     3
3  0.592593    0.592593         0     4
4  0.611111    0.611111         0     5
exp_acc       0.625173
strong_acc    0.625173
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 3/4 with seed 44 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.655172    0.655172         0     1
1  0.62

In [ ]:
from main_ex import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["fixation_time", "N100"]
all_dfs = []
repeats = 4
for r in range(repeats):
    seed = 42 + r  
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,

    )
 
    res_df["repeat"] = r + 1
    all_dfs.append(res_df)

df_pdt = pd.concat(all_dfs, ignore_index=True)
summary = df_pdt.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/4 with seed 42 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.620690    0.620690         0     1
1  0.791667    0.791667         0     2
2  0.705882    0.705882         0     3
3  0.666667    0.666667         0     4
4  0.611111    0.611111         0     5
exp_acc       0.679203
strong_acc    0.679203
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 2/4 with seed 43 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.586207    0.586207         0     1
1  0.625000    0.625000         0     2
2  0.705882    0.705882         0     3
3  0.629630    0.629630         0     4
4  0.694444    0.694444         0     5
exp_acc       0.648233
strong_acc    0.648233
weak_acc      0.000000
fold          3.000000
dtype: float64

===== Running repeat 3/4 with seed 44 =====

===== K-Fold Summary =====
    exp_acc  strong_acc  weak_acc  fold
0  0.620690    0.620690         0     1
1  0.54

In [40]:
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, df_eye["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"90% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, df_n100["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"90% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, df_ft["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"90% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, df_pdt["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"90% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")


[ExpAcc] Gaze vs Baseline: Δ=0.0352, p=0.0390
90% CI: (0.003, 0.069)
[ExpAcc] Gaze vs Baseline: Δ=0.0381, p=0.0460
90% CI: (0.001, 0.071)
[ExpAcc] Gaze vs Baseline: Δ=0.0315, p=0.0570
90% CI: (-0.001, 0.063)
[ExpAcc] Gaze vs Baseline: Δ=0.0310, p=0.1020
90% CI: (-0.008, 0.065)


In [ ]:
from pairwise_tie import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed", "fixation_time", "N100"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 38 + r   
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=[],
        use_gaze=False,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)


df = pd.concat(all_dfs, ignore_index=True)
summary = df.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 38 =====
Fold 1: WeightedAcc=0.496, ExpAcc=0.490, StrongAcc=0.448, WeakAcc=0.507
Fold 2: WeightedAcc=0.663, ExpAcc=0.490, StrongAcc=0.792, WeakAcc=0.392
Fold 3: WeightedAcc=0.643, ExpAcc=0.588, StrongAcc=0.647, WeakAcc=0.556
Fold 4: WeightedAcc=0.532, ExpAcc=0.485, StrongAcc=0.481, WeakAcc=0.486
Fold 5: WeightedAcc=0.534, ExpAcc=0.443, StrongAcc=0.667, WeakAcc=0.311

===== K-Fold Summary =====
fold          3.000000
w_acc         0.573686
exp_acc       0.499011
strong_acc    0.607030
weak_acc      0.450377
dtype: float64

===== Running repeat 2/3 with seed 39 =====
Fold 1: WeightedAcc=0.513, ExpAcc=0.480, StrongAcc=0.483, WeakAcc=0.478
Fold 2: WeightedAcc=0.624, ExpAcc=0.480, StrongAcc=0.708, WeakAcc=0.405
Fold 3: WeightedAcc=0.661, ExpAcc=0.577, StrongAcc=0.676, WeakAcc=0.524
Fold 4: WeightedAcc=0.624, ExpAcc=0.567, StrongAcc=0.630, WeakAcc=0.543
Fold 5: WeightedAcc=0.534, ExpAcc=0.443, StrongAcc=0.667, WeakAcc=0.311

===== K-Fold Summary =====
fold

In [ ]:
from pairwise_tie import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed", "fixation_time", "N100"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 38 + r  
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)


dfeye = pd.concat(all_dfs, ignore_index=True)
summary = dfeye.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 38 =====
Fold 1: WeightedAcc=0.531, ExpAcc=0.490, StrongAcc=0.552, WeakAcc=0.464
Fold 2: WeightedAcc=0.713, ExpAcc=0.541, StrongAcc=0.833, WeakAcc=0.446
Fold 3: WeightedAcc=0.643, ExpAcc=0.588, StrongAcc=0.618, WeakAcc=0.571
Fold 4: WeightedAcc=0.642, ExpAcc=0.567, StrongAcc=0.667, WeakAcc=0.529
Fold 5: WeightedAcc=0.517, ExpAcc=0.443, StrongAcc=0.639, WeakAcc=0.328

===== K-Fold Summary =====
fold          3.000000
w_acc         0.609295
exp_acc       0.525710
strong_acc    0.661652
weak_acc      0.467517
dtype: float64

===== Running repeat 2/3 with seed 39 =====
Fold 1: WeightedAcc=0.549, ExpAcc=0.531, StrongAcc=0.517, WeakAcc=0.536
Fold 2: WeightedAcc=0.673, ExpAcc=0.500, StrongAcc=0.792, WeakAcc=0.405
Fold 3: WeightedAcc=0.661, ExpAcc=0.588, StrongAcc=0.647, WeakAcc=0.556
Fold 4: WeightedAcc=0.661, ExpAcc=0.577, StrongAcc=0.667, WeakAcc=0.543
Fold 5: WeightedAcc=0.568, ExpAcc=0.464, StrongAcc=0.694, WeakAcc=0.328

===== K-Fold Summary =====
fold

In [ ]:
from pairwise_tie import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed", "fixation_time"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 38 + r   
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)

dfn100 = pd.concat(all_dfs, ignore_index=True)
summary = dfn100.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 38 =====
Fold 1: WeightedAcc=0.584, ExpAcc=0.561, StrongAcc=0.552, WeakAcc=0.565
Fold 2: WeightedAcc=0.644, ExpAcc=0.500, StrongAcc=0.750, WeakAcc=0.419
Fold 3: WeightedAcc=0.652, ExpAcc=0.577, StrongAcc=0.618, WeakAcc=0.556
Fold 4: WeightedAcc=0.560, ExpAcc=0.505, StrongAcc=0.593, WeakAcc=0.471
Fold 5: WeightedAcc=0.534, ExpAcc=0.433, StrongAcc=0.639, WeakAcc=0.311

===== K-Fold Summary =====
fold          3.000000
w_acc         0.594668
exp_acc       0.515338
strong_acc    0.630171
weak_acc      0.464519
dtype: float64

===== Running repeat 2/3 with seed 39 =====
Fold 1: WeightedAcc=0.496, ExpAcc=0.469, StrongAcc=0.483, WeakAcc=0.464
Fold 2: WeightedAcc=0.693, ExpAcc=0.520, StrongAcc=0.833, WeakAcc=0.419
Fold 3: WeightedAcc=0.609, ExpAcc=0.557, StrongAcc=0.588, WeakAcc=0.540
Fold 4: WeightedAcc=0.606, ExpAcc=0.515, StrongAcc=0.630, WeakAcc=0.471
Fold 5: WeightedAcc=0.534, ExpAcc=0.423, StrongAcc=0.667, WeakAcc=0.279

===== K-Fold Summary =====
fold

In [ ]:
from pairwise_tie import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["pixel_speed",  "N100"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 38 + r   
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)

dfft = pd.concat(all_dfs, ignore_index=True)
summary = dfft.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 38 =====


Fold 1: WeightedAcc=0.566, ExpAcc=0.541, StrongAcc=0.552, WeakAcc=0.536
Fold 2: WeightedAcc=0.663, ExpAcc=0.500, StrongAcc=0.792, WeakAcc=0.405
Fold 3: WeightedAcc=0.652, ExpAcc=0.567, StrongAcc=0.618, WeakAcc=0.540
Fold 4: WeightedAcc=0.560, ExpAcc=0.495, StrongAcc=0.593, WeakAcc=0.457
Fold 5: WeightedAcc=0.542, ExpAcc=0.443, StrongAcc=0.667, WeakAcc=0.311

===== K-Fold Summary =====
fold          3.000000
w_acc         0.596784
exp_acc       0.509194
strong_acc    0.644059
weak_acc      0.449988
dtype: float64

===== Running repeat 2/3 with seed 39 =====
Fold 1: WeightedAcc=0.522, ExpAcc=0.500, StrongAcc=0.517, WeakAcc=0.493
Fold 2: WeightedAcc=0.683, ExpAcc=0.541, StrongAcc=0.792, WeakAcc=0.459
Fold 3: WeightedAcc=0.635, ExpAcc=0.577, StrongAcc=0.618, WeakAcc=0.556
Fold 4: WeightedAcc=0.578, ExpAcc=0.495, StrongAcc=0.593, WeakAcc=0.457
Fold 5: WeightedAcc=0.534, ExpAcc=0.454, StrongAcc=0.639, WeakAcc=0.344

===== K-Fold Summary =====
fold          3.000000
w_acc         0.590391
exp

In [ ]:
from pairwise_tie import run_kfold_cv
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS = ["N100","fixation_time"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 38 + r 
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_cv(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1  
    all_dfs.append(res_df)

dfpdt = pd.concat(all_dfs, ignore_index=True)
summary = dfpdt.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 38 =====
Fold 1: WeightedAcc=0.566, ExpAcc=0.531, StrongAcc=0.552, WeakAcc=0.522
Fold 2: WeightedAcc=0.673, ExpAcc=0.510, StrongAcc=0.792, WeakAcc=0.419


Fold 3: WeightedAcc=0.652, ExpAcc=0.588, StrongAcc=0.618, WeakAcc=0.571
Fold 4: WeightedAcc=0.596, ExpAcc=0.536, StrongAcc=0.630, WeakAcc=0.500
Fold 5: WeightedAcc=0.542, ExpAcc=0.443, StrongAcc=0.639, WeakAcc=0.328

===== K-Fold Summary =====
fold          3.000000
w_acc         0.606103
exp_acc       0.521565
strong_acc    0.645911
weak_acc      0.467991
dtype: float64

===== Running repeat 2/3 with seed 39 =====
Fold 1: WeightedAcc=0.504, ExpAcc=0.510, StrongAcc=0.483, WeakAcc=0.522
Fold 2: WeightedAcc=0.703, ExpAcc=0.541, StrongAcc=0.833, WeakAcc=0.446
Fold 3: WeightedAcc=0.652, ExpAcc=0.598, StrongAcc=0.618, WeakAcc=0.587
Fold 4: WeightedAcc=0.587, ExpAcc=0.505, StrongAcc=0.593, WeakAcc=0.471
Fold 5: WeightedAcc=0.525, ExpAcc=0.423, StrongAcc=0.667, WeakAcc=0.279

===== K-Fold Summary =====
fold          3.000000
w_acc         0.594430
exp_acc       0.515359
strong_acc    0.638600
weak_acc      0.461021
dtype: float64

===== Running repeat 3/3 with seed 40 =====
Fold 1: WeightedAc

In [21]:
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["weak_acc"].values, dfeye["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["weak_acc"].values, dfn100["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["weak_acc"].values, dfft["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["weak_acc"].values, dfpdt["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")


[ExpAcc] Gaze vs Baseline: Δ=0.0037, p=0.3660
95% CI: (-0.020, 0.025)
[ExpAcc] Gaze vs Baseline: Δ=0.0079, p=0.2310
95% CI: (-0.012, 0.029)
[ExpAcc] Gaze vs Baseline: Δ=0.0105, p=0.1720
95% CI: (-0.011, 0.032)
[ExpAcc] Gaze vs Baseline: Δ=0.0104, p=0.1720
95% CI: (-0.011, 0.030)


In [22]:
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, dfeye["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, dfn100["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, dfft["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["strong_acc"].values, dfpdt["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")


[ExpAcc] Gaze vs Baseline: Δ=0.0310, p=0.0110
95% CI: (0.004, 0.060)
[ExpAcc] Gaze vs Baseline: Δ=0.0086, p=0.3330
95% CI: (-0.031, 0.043)
[ExpAcc] Gaze vs Baseline: Δ=0.0178, p=0.1040
95% CI: (-0.009, 0.044)
[ExpAcc] Gaze vs Baseline: Δ=0.0156, p=0.2190
95% CI: (-0.020, 0.052)


In [23]:
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["exp_acc"].values, dfeye["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["exp_acc"].values, dfn100["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["exp_acc"].values, dfft["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df["exp_acc"].values, dfpdt["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")


[ExpAcc] Gaze vs Baseline: Δ=0.0110, p=0.1170
95% CI: (-0.005, 0.029)
[ExpAcc] Gaze vs Baseline: Δ=0.0081, p=0.2270
95% CI: (-0.012, 0.029)
[ExpAcc] Gaze vs Baseline: Δ=0.0119, p=0.1320
95% CI: (-0.008, 0.031)
[ExpAcc] Gaze vs Baseline: Δ=0.0112, p=0.1660
95% CI: (-0.011, 0.031)


In [ ]:
from regression import run_kfold_regression
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS =  ["pixel_speed", "fixation_time", "N100"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 44 + r  
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_regression(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=False,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)

df_re = pd.concat(all_dfs, ignore_index=True)
summary = df_re.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 44 =====
Fold 1: WeightedAcc=0.522, ExpAcc=0.459, StrongAcc=0.483, WeakAcc=0.449
Fold 2: WeightedAcc=0.683, ExpAcc=0.500, StrongAcc=0.833, WeakAcc=0.392
Fold 3: WeightedAcc=0.609, ExpAcc=0.505, StrongAcc=0.618, WeakAcc=0.444
Fold 4: WeightedAcc=0.624, ExpAcc=0.546, StrongAcc=0.630, WeakAcc=0.514
Fold 5: WeightedAcc=0.542, ExpAcc=0.412, StrongAcc=0.667, WeakAcc=0.262

=== K-Fold Summary ===
fold          3.000000
w_acc         0.596043
exp_acc       0.484620
strong_acc    0.646007
weak_acc      0.412438
dtype: float64

===== Running repeat 2/3 with seed 45 =====
Fold 1: WeightedAcc=0.558, ExpAcc=0.480, StrongAcc=0.552, WeakAcc=0.449
Fold 2: WeightedAcc=0.624, ExpAcc=0.459, StrongAcc=0.750, WeakAcc=0.365
Fold 3: WeightedAcc=0.670, ExpAcc=0.567, StrongAcc=0.676, WeakAcc=0.508
Fold 4: WeightedAcc=0.560, ExpAcc=0.485, StrongAcc=0.556, WeakAcc=0.457
Fold 5: WeightedAcc=0.534, ExpAcc=0.433, StrongAcc=0.639, WeakAcc=0.311

=== K-Fold Summary ===
fold        

In [ ]:
from regression import run_kfold_regression
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS =  ["pixel_speed", "fixation_time", "N100"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 44 + r  
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_regression(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1  
    all_dfs.append(res_df)


df_reeye = pd.concat(all_dfs, ignore_index=True)
summary = df_reeye.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 44 =====
Fold 1: WeightedAcc=0.584, ExpAcc=0.551, StrongAcc=0.517, WeakAcc=0.565
Fold 2: WeightedAcc=0.614, ExpAcc=0.449, StrongAcc=0.750, WeakAcc=0.351
Fold 3: WeightedAcc=0.635, ExpAcc=0.536, StrongAcc=0.647, WeakAcc=0.476
Fold 4: WeightedAcc=0.523, ExpAcc=0.454, StrongAcc=0.519, WeakAcc=0.429
Fold 5: WeightedAcc=0.525, ExpAcc=0.412, StrongAcc=0.639, WeakAcc=0.279

=== K-Fold Summary ===
fold          3.000000
w_acc         0.576215
exp_acc       0.480412
strong_acc    0.614342
weak_acc      0.420004
dtype: float64

===== Running repeat 2/3 with seed 45 =====
Fold 1: WeightedAcc=0.487, ExpAcc=0.439, StrongAcc=0.448, WeakAcc=0.435
Fold 2: WeightedAcc=0.703, ExpAcc=0.520, StrongAcc=0.875, WeakAcc=0.405
Fold 3: WeightedAcc=0.643, ExpAcc=0.557, StrongAcc=0.618, WeakAcc=0.524
Fold 4: WeightedAcc=0.532, ExpAcc=0.464, StrongAcc=0.519, WeakAcc=0.443
Fold 5: WeightedAcc=0.568, ExpAcc=0.454, StrongAcc=0.667, WeakAcc=0.328

=== K-Fold Summary ===
fold        

In [ ]:
from regression import run_kfold_regression
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS =  ["pixel_speed", "fixation_time"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 44 + r   
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_regression(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1  
    all_dfs.append(res_df)

df_ren100 = pd.concat(all_dfs, ignore_index=True)
summary = df_ren100.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 44 =====
Fold 1: WeightedAcc=0.540, ExpAcc=0.500, StrongAcc=0.517, WeakAcc=0.493
Fold 2: WeightedAcc=0.673, ExpAcc=0.520, StrongAcc=0.708, WeakAcc=0.459
Fold 3: WeightedAcc=0.661, ExpAcc=0.567, StrongAcc=0.647, WeakAcc=0.524
Fold 4: WeightedAcc=0.560, ExpAcc=0.495, StrongAcc=0.556, WeakAcc=0.471
Fold 5: WeightedAcc=0.559, ExpAcc=0.454, StrongAcc=0.667, WeakAcc=0.328

=== K-Fold Summary ===
fold          3.000000
w_acc         0.598583
exp_acc       0.507174
strong_acc    0.618971
weak_acc      0.455064
dtype: float64

===== Running repeat 2/3 with seed 45 =====
Fold 1: WeightedAcc=0.531, ExpAcc=0.480, StrongAcc=0.517, WeakAcc=0.464
Fold 2: WeightedAcc=0.604, ExpAcc=0.480, StrongAcc=0.625, WeakAcc=0.432
Fold 3: WeightedAcc=0.609, ExpAcc=0.515, StrongAcc=0.618, WeakAcc=0.460
Fold 4: WeightedAcc=0.596, ExpAcc=0.505, StrongAcc=0.593, WeakAcc=0.471
Fold 5: WeightedAcc=0.593, ExpAcc=0.464, StrongAcc=0.722, WeakAcc=0.311

=== K-Fold Summary ===
fold        

In [ ]:
from regression import run_kfold_regression
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS =  [ "fixation_time", "N100"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 44 + r  
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_regression(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)


df_repdt = pd.concat(all_dfs, ignore_index=True)
summary = df_repdt.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 44 =====
Fold 1: WeightedAcc=0.531, ExpAcc=0.459, StrongAcc=0.552, WeakAcc=0.420
Fold 2: WeightedAcc=0.673, ExpAcc=0.510, StrongAcc=0.792, WeakAcc=0.419
Fold 3: WeightedAcc=0.652, ExpAcc=0.557, StrongAcc=0.618, WeakAcc=0.524
Fold 4: WeightedAcc=0.523, ExpAcc=0.454, StrongAcc=0.556, WeakAcc=0.414
Fold 5: WeightedAcc=0.551, ExpAcc=0.433, StrongAcc=0.667, WeakAcc=0.295

=== K-Fold Summary ===
fold          3.000000
w_acc         0.586040
exp_acc       0.482537
strong_acc    0.636652
weak_acc      0.414477
dtype: float64

===== Running repeat 2/3 with seed 45 =====
Fold 1: WeightedAcc=0.558, ExpAcc=0.500, StrongAcc=0.517, WeakAcc=0.493
Fold 2: WeightedAcc=0.703, ExpAcc=0.520, StrongAcc=0.875, WeakAcc=0.405
Fold 3: WeightedAcc=0.643, ExpAcc=0.546, StrongAcc=0.647, WeakAcc=0.492
Fold 4: WeightedAcc=0.606, ExpAcc=0.515, StrongAcc=0.630, WeakAcc=0.471
Fold 5: WeightedAcc=0.551, ExpAcc=0.443, StrongAcc=0.694, WeakAcc=0.295

=== K-Fold Summary ===
fold        

In [ ]:
from regression import run_kfold_regression
import torch
import pandas as pd
DATA_PATH = "metrics_collection_pca256.csv"
GAZE_COLS =  ["pixel_speed", "N100"]
all_dfs = []
repeats = 3
for r in range(repeats):
    seed = 44 + r  
 
    set_seed(seed)
    print(f"\n===== Running repeat {r+1}/{repeats} with seed {seed} =====")
    res_df = run_kfold_regression(
        data_path=DATA_PATH,
        gaze_cols=GAZE_COLS,
        use_gaze=True,
        k_folds=5,
        hidden_dim=64,
        dropout=0.3,
        lr=5e-5,
        batch_size=32,
        epochs=40,
        device="cuda"
    )
 
    res_df["repeat"] = r + 1 
    all_dfs.append(res_df)

df_reft = pd.concat(all_dfs, ignore_index=True)
summary = df_reft.groupby("repeat").mean(numeric_only=True).describe().loc[["mean", "std"]]
print("\n===== Repeated K-Fold Summary =====")
print(summary)


===== Running repeat 1/3 with seed 44 =====
Fold 1: WeightedAcc=0.575, ExpAcc=0.500, StrongAcc=0.552, WeakAcc=0.478
Fold 2: WeightedAcc=0.653, ExpAcc=0.500, StrongAcc=0.750, WeakAcc=0.419
Fold 3: WeightedAcc=0.652, ExpAcc=0.577, StrongAcc=0.618, WeakAcc=0.556
Fold 4: WeightedAcc=0.560, ExpAcc=0.464, StrongAcc=0.593, WeakAcc=0.414
Fold 5: WeightedAcc=0.559, ExpAcc=0.454, StrongAcc=0.667, WeakAcc=0.328

=== K-Fold Summary ===
fold          3.000000
w_acc         0.599963
exp_acc       0.498969
strong_acc    0.635726
weak_acc      0.438978
dtype: float64

===== Running repeat 2/3 with seed 45 =====
Fold 1: WeightedAcc=0.549, ExpAcc=0.480, StrongAcc=0.517, WeakAcc=0.464
Fold 2: WeightedAcc=0.683, ExpAcc=0.490, StrongAcc=0.875, WeakAcc=0.365
Fold 3: WeightedAcc=0.652, ExpAcc=0.546, StrongAcc=0.676, WeakAcc=0.476
Fold 4: WeightedAcc=0.578, ExpAcc=0.485, StrongAcc=0.593, WeakAcc=0.443
Fold 5: WeightedAcc=0.585, ExpAcc=0.464, StrongAcc=0.722, WeakAcc=0.311

=== K-Fold Summary ===
fold        

In [24]:
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["weak_acc"].values, df_reeye["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["weak_acc"].values, df_ren100["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["weak_acc"].values, df_reft["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["weak_acc"].values, df_repdt["weak_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")


[ExpAcc] Gaze vs Baseline: Δ=0.0112, p=0.1710
95% CI: (-0.013, 0.034)
[ExpAcc] Gaze vs Baseline: Δ=0.0242, p=0.0240
95% CI: (0.000, 0.047)
[ExpAcc] Gaze vs Baseline: Δ=0.0115, p=0.2010
95% CI: (-0.016, 0.039)
[ExpAcc] Gaze vs Baseline: Δ=0.0157, p=0.1050
95% CI: (-0.010, 0.040)


In [25]:
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["exp_acc"].values, df_reeye["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["exp_acc"].values, df_ren100["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["exp_acc"].values, df_reft["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["exp_acc"].values, df_repdt["exp_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")


[ExpAcc] Gaze vs Baseline: Δ=0.0023, p=0.4340
95% CI: (-0.019, 0.024)
[ExpAcc] Gaze vs Baseline: Δ=0.0154, p=0.0780
95% CI: (-0.006, 0.034)
[ExpAcc] Gaze vs Baseline: Δ=0.0120, p=0.1470
95% CI: (-0.009, 0.032)
[ExpAcc] Gaze vs Baseline: Δ=0.0167, p=0.0770
95% CI: (-0.005, 0.038)


In [26]:
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["strong_acc"].values, df_reeye["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["strong_acc"].values, df_ren100["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["strong_acc"].values, df_reft["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")
mean_diff, p_value, ci_range = bootstrap_significance_ci(df_re["strong_acc"].values, df_repdt["strong_acc"].values)
print(f"[ExpAcc] Gaze vs Baseline: Δ={mean_diff:.4f}, p={p_value:.4f}")
print(f"95% CI: ({ci_range[0]:.3f}, {ci_range[1]:.3f})")


[ExpAcc] Gaze vs Baseline: Δ=-0.0130, p=0.7630
95% CI: (-0.047, 0.024)
[ExpAcc] Gaze vs Baseline: Δ=-0.0082, p=0.6750
95% CI: (-0.044, 0.028)
[ExpAcc] Gaze vs Baseline: Δ=0.0142, p=0.1880
95% CI: (-0.016, 0.045)
[ExpAcc] Gaze vs Baseline: Δ=0.0205, p=0.1110
95% CI: (-0.012, 0.055)


In [ ]:
import pandas as pd
explist_df = pd.read_csv('exp_list_cleaned.csv')
unique_df = pd.read_csv('unique_conversations.csv')

rows = []
for _, row in explist_df.iterrows():
    for conv, suffix in [("conversation-a", "_a"), ("conversation-b", "_b")]:
        conv_id = row[conv]
        match = unique_df[unique_df["text"] == conv_id]
        if not match.empty:
            sentence_id = match.iloc[0]["sentenceid"]
            emb_path = match.iloc[0]["embedding_path"]
            rows.append({
                f"sentenceid{suffix}": sentence_id,
                f"embedding_path{suffix}": emb_path
            })
        else:
            rows.append({
                f"sentenceid{suffix}": None,
                f"embedding_path{suffix}": None
            })


for i, info in enumerate(rows[::2]):
    explist_df.loc[i, "sentenceid_a"] = info["sentenceid_a"]
    explist_df.loc[i, "embedding_path_a"] = info["embedding_path_a"]
    explist_df.loc[i, "sentenceid_b"] = rows[2*i+1]["sentenceid_b"]
    explist_df.loc[i, "embedding_path_b"] = rows[2*i+1]["embedding_path_b"]

explist_df.to_csv('exp_list_cleaned.csv', index=False)


In [ ]:

import pandas as pd
explist_df = pd.read_csv('exp_list_cleaned.csv')
metrics_df = pd.read_csv('metrics_collection.csv')


metrics_df = metrics_df.groupby('pair_id').filter(lambda x: len(x) == 2)


for idx, row in metrics_df.iterrows():
    exp_row = explist_df[explist_df['id'] == row['pair_id']]
    if exp_row.empty:
        continue
    if row['conversation'] == 1:
        metrics_df.at[idx, 'text'] = exp_row.iloc[0]['conversation-a']
        metrics_df.at[idx, 'text_id'] = exp_row.iloc[0]['sentenceid_a']
        metrics_df.at[idx, 'embedding_path'] = exp_row.iloc[0]['embedding_path_a']
    elif row['conversation'] == 2:
        metrics_df.at[idx, 'text'] = exp_row.iloc[0]['conversation-b']
        metrics_df.at[idx, 'text_id'] = exp_row.iloc[0]['sentenceid_b']
        metrics_df.at[idx, 'embedding_path'] = exp_row.iloc[0]['embedding_path_b']



metrics_df.to_csv('metrics_collection.csv', index=False)


In [1]:
import pandas as pd

df = pd.read_csv('metrics_collection.csv')
count = ((df['exp'] == 1) | (df['exp'] == 2)).sum()
proportion = count / len(df)

print(f"Count of exp == 1.5: {count}")
print(f"Proportion of exp == 1.5: {proportion:.2%}")

Count of exp == 1.5: 300
Proportion of exp == 1.5: 30.80%
